In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ito at maaaring magbago.

## Pangkalahatang-ideya
Ang IBM&reg; Circuit function ay tumatanggap ng [abstract PUBs](./primitive-input-output) bilang mga input, at nagbabalik ng mitigated na expectation values bilang mga output. Kasama sa circuit function na ito ang isang automated at customized na pipeline para makapag-focus ang mga mananaliksik sa pag-discover ng algorithm at application.

## Paglalarawan
Pagkatapos mag-submit ng PUB, ang iyong mga abstract circuit at observable ay awtomatikong tina-transpile, pinatatakbo sa hardware, at nipo-post-process para ibalik ang mitigated na expectation values. Para magawa ito, pinagsama ang mga sumusunod na kagamitan:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), kasama ang auto-selection ng AI-driven at heuristic na transpilation passes para i-translate ang iyong mga abstract circuit sa hardware-optimized na ISA circuits
- [Error suppression at mitigation na kailangan para sa utility-scale na computation](./error-mitigation-and-suppression-techniques), kasama ang measurement at gate twirling, dynamical decoupling, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE), at Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), para patakbuhin ang mga ISA PUB sa hardware at ibalik ang mitigated na expectation values

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## Pagsisimula
Mag-authenticate gamit ang iyong [API key](http://quantum.cloud.ibm.com/) at piliin ang Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na nai-save mo na ang iyong account [sa iyong lokal na environment](/guides/functions#install-qiskit-functions-catalog-client).)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Halimbawa
Para makapagsimula, subukan ang basic na halimbawang ito:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Tingnan ang [status](/guides/functions#check-job-status) ng iyong Qiskit Function workload o kunin ang [mga resulta](/guides/functions#retrieve-results) tulad ng sumusunod:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Ang mga resulta ay nasa parehong format ng isang [Estimator result](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Mga Input
Tingnan ang sumusunod na talahanayan para sa lahat ng input parameter na tinatanggap ng IBM Circuit function. Ang susunod na seksyon ng [Mga Opsyon](#options) ay nagbibigay ng mas detalyadong impormasyon tungkol sa mga available na `options`.

| Pangalan      | Uri                       | Paglalarawan                                                                                                                                                                                                                         | Kinakailangan | Default                                                                  | Halimbawa                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Pangalan ng backend para sa query.                                                                                                                                                                                              | Oo      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Isang iterable ng abstract PUB-like (primitive unified bloc) na mga object, tulad ng mga tuple na `(circuit, observables)` o `(circuit, observables, parameter_values)`. Tingnan ang [Pangkalahatang-ideya ng PUBs](/guides/primitive-input-output#overview-of-pubs) para sa karagdagang impormasyon. Ang mga circuit ay maaaring abstract (non-ISA). | Oo      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Mga input na opsyon. Tingnan ang seksyon ng **Mga Opsyon** para sa karagdagang detalye.                                                                                                                                                                                | Hindi       | Tingnan ang seksyon ng **Mga Opsyon** para sa mga detalye.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Ang cloud resource name ng instance na gagamitin sa format na iyon.                                                                                                                                                                                        | Hindi       | Isang random ang pipiliin kung ang iyong account ay may access sa maramihang instance. | `CRN`                   |

### Mga Opsyon
#### Istruktura
Katulad ng Qiskit Runtime primitives, ang mga opsyon para sa IBM Circuit function ay maaaring tukuyin bilang isang nested dictionary. Ang mga karaniwang ginagamit na opsyon, tulad ng ``optimization_level`` at ``mitigation_level``, ay nasa unang antas. Ang iba pang mas advanced na opsyon ay nakagrupo sa iba't ibang kategorya, tulad ng ``resilience``.

#### Mga Default
Kung hindi ka nagtukoy ng value para sa isang opsyon, gagamitin ang default na value na itinakda ng serbisyo.

#### Antas ng mitigation
Sinusuportahan din ng IBM Circuit function ang `mitigation_level`. Tinutukoy ng mitigation level kung gaano karaming error suppression at mitigation ang ilalapat sa job. Ang mas mataas na antas ay nagbibigay ng mas tumpak na mga resulta, ngunit mas matagal ang processing time. Ang antas ng pagbabawas ng error ay depende sa pamamaraang ginamit. Binibigyan ng abstraction ng mitigation level ang detalyadong pagpili ng mga paraan ng error mitigation at suppression para makapag-isip ang mga gumagamit tungkol sa cost/accuracy trade-off na angkop sa kanilang application. Ipinapakita ng sumusunod na talahanayan ang mga katumbas na pamamaraan para sa bawat antas.

> **Note:** Kahit magkapareho ang mga pangalan, ang `mitigation_level` ay gumagamit ng ibang mga teknik kaysa sa ginagamit ng `resilience_level` ng Estimator.

Katulad ng ``resilience_level`` sa Qiskit Runtime Estimator, tinutukoy ng ``mitigation_level`` ang isang base set ng mga curated na opsyon. Ang anumang opsyon na mano-manong tinukoy mo bukod sa mitigation level ay ilalapat sa ibabaw ng base set ng mga opsyon na tinukoy ng mitigation level. Kaya naman, sa prinsipyo, maaari mong itakda ang mitigation level sa 1, ngunit pagkatapos ay i-off ang measurement mitigation, kahit hindi ito inirerekomenda.

| **Mitigation Level** | **Teknik** |
|:-:|:-:|
| 1 [Default] | Dynamical decoupling + measurement twirling + TREX  |
| 2 | Level 1 + gate twirling + ZNE via gate folding |
| 3 | Level 1 + gate twirling + ZNE via PEA |

Ipinapakita ng sumusunod na halimbawa ang pagtatakda ng mitigation level:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Lahat ng available na opsyon
Bukod sa `mitigation_level`, nagbibigay din ang IBM Circuit function ng ilang advanced na opsyon na nagbibigay-daan sa iyo na i-fine-tune ang cost/accuracy trade-off. Ipinapakita ng sumusunod na talahanayan ang lahat ng available na opsyon:

| Opsyon | Sub-opsyon | Sub-sub-opsyon | Paglalarawan | Mga Pagpipilian | Default |
|-|-|-|-|-|-|
| default_precision |  |  | Ang default na precision na gagamitin para sa anumang PUB o `run()`<br/>call na hindi nagtutukoy ng isa.<br/>Ang bawat input PUB ay maaaring magtukoy ng sariling precision. Kung ang `run()` method ay binibigyan ng precision, ang value na iyon ang gagamitin para sa lahat ng PUB sa `run()` call na hindi nagtutukoy ng sarili nilang precision.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum na oras ng execution sa segundo, batay sa<br/>QPU usage (hindi wall clock time). Ang QPU usage ay ang<br/>dami ng oras na nakalaan ang QPU sa pagpoproseso ng iyong job. Kung lalampas ang isang job sa time limit na ito, ito ay puwersahang icancelled. | Integer na bilang ng segundo sa range na [1, 10800] |  |
| mitigation_level |  |  | Gaano karaming error suppression at mitigation ang ilalapat. Sumangguni sa seksyon ng [Antas ng mitigation](#mitigation-level) para sa karagdagang impormasyon tungkol sa mga pamamaraang ginagamit sa bawat antas. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Gaano karaming optimization ang isasagawa sa mga circuit. Ang [Mas mataas na antas](/guides/set-optimization) ay nagge-generate ng mas na-optimize na mga circuit, ngunit mas matagal ang transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Kung ie-enable ang dynamical decoupling. Sumangguni sa [Mga teknik ng error suppression at mitigation](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) para sa paliwanag ng pamamaraan.  | True/False | True |
|  | sequence_type |  | Kung aling dynamical decoupling sequence ang gagamitin.<br/>* `XX`: gamitin ang sequence na `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: gamitin ang sequence na `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: gamitin ang sequence na<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Kung ilalapat ang 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Kung ie-enable ang twirling ng mga measurement. | True/False | True |
| resilience | measure_mitigation |  | Kung ie-enable ang TREX measurement error mitigation method. Sumangguni sa [Mga teknik ng error suppression at mitigation](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) para sa paliwanag ng pamamaraan.  | True/False | True |
|  | zne_mitigation |  | Kung io-on ang Zero Noise Extrapolation error mitigation method. Sumangguni sa [Mga teknik ng error suppression at mitigation](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) para sa paliwanag ng pamamaraan.  | True/False | False |
|  | zne | amplifier | Kung aling teknik ang gagamitin para sa pagpapalaki ng noise. Isa sa: <br/> - `gate_folding` (default) gumagamit ng 2-qubit gate folding para palakihin ang noise. Kung ang noise factor ay nangangailangan na palakihin lamang ang isang subset ng mga gate, ang mga gate na ito ay random na pinipili.<br/><br/> - `gate_folding_front` gumagamit ng 2-qubit gate folding para palakihin ang noise. Kung ang noise factor ay nangangailangan na palakihin lamang ang isang subset ng mga gate, ang mga gate na ito ay pinipili mula sa harapan ng topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` gumagamit ng 2-qubit gate folding para palakihin ang noise. Kung ang noise factor ay nangangailangan na palakihin lamang ang isang subset ng mga gate, ang mga gate na ito ay pinipili mula sa likod ng topologically ordered DAG circuit.<br/><br/> - `pea` gumagamit ng teknik na tinatawag na Probabilistic error amplification (PEA) para palakihin ang noise. Sumangguni sa [Mga teknik ng error suppression at mitigation](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) para sa paliwanag ng pamamaraan.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Mga noise factor na gagamitin para sa noise amplification. | listahan ng mga float; bawat float >= 1 | (1, 1.5, 2) para sa PEA, at (1, 3, 5) para sa iba. |
|  |  | extrapolator | Mga noise factor para sa pag-evaluate ng fit extrapolation models. Ang opsyon na ito ay hindi nakakaapekto sa execution o model fitting sa anumang paraan; tinutukoy lamang nito ang mga punto kung saan ine-evaluate ang mga `extrapolator` object para ibalik sa mga data field na tinatawag na `evs_extrapolated` at `stds_extrapolated`. | isa o higit pa sa `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Kung io-on ang Probabilistic Error Cancellation error mitigation method. Sumangguni sa [Mga teknik ng error suppression at mitigation](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) para sa paliwanag ng pamamaraan.  | True/False | False |
|  | pec | max_overhead | Ang maximum na circuit sampling overhead na pinahihintulutan, o `None` para walang maximum. | None/ integer >1 | 100 |

Sa sumusunod na halimbawa, ang pagtatakda ng mitigation level sa 1 ay unang nag-a-off ng ZNE mitigation, ngunit ang pagtatakda ng `zne_mitigation` sa `True` ay nino-override ang kaugnay na setup mula sa `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Mga Output
Ang output ng isang Circuit function ay isang [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), na naglalaman ng dalawang field:

- Isa o higit pang [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na mga object. Maaaring i-index ang mga ito nang direkta mula sa `PrimitiveResult`.

- Job-level na metadata.

Ang bawat `PubResult` ay naglalaman ng `data` at `metadata` na field.

- Ang `data` field ay naglalaman ng hindi bababa sa isang array ng mga expectation value (`PubResult.data.evs`) at isang array ng mga standard error (`PubResult.data.stds`). Maaari rin itong maglaman ng karagdagang data, depende sa mga opsyon na ginamit.

- Ang `metadata` field ay naglalaman ng PUB-level na metadata (`PubResult.metadata`).

Inilalarawan ng sumusunod na code snippet ang format ng `PrimitiveResult` (at kaugnay na `PubResult`).